# NIH45-46 matched-pair scatters

Matched pairs of genotypes differing only at one focal site, with the unmutated background's affinity on x and the mutated genotype's on y, so displacement from the diagonal is effect of that substitution in that background. Panels stratify by antigen and by a conditioning site; color gives the state of two modifier sites.

## 1. Setup

In [ ]:
import os
import colorsys
import itertools
import re
from pathlib import Path

import matplotlib
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.patches as mpatches
import matplotlib.ticker as mticker
from matplotlib.colors import to_rgb, to_hex
from matplotlib.patches import Patch
from matplotlib.lines import Line2D
from matplotlib.ticker import MaxNLocator
import numpy as np
import pandas as pd
from scipy.stats import gaussian_kde

%matplotlib inline

## 2. Paths and constants

In [ ]:
# ---- paths -------------------------------------------------------------
data_dir = "../data"
fig_dir = "../figures"
table_dir = "../tables"
os.makedirs(fig_dir, exist_ok=True)
os.makedirs(table_dir, exist_ok=True)
PNG_DPI = 300

plt.rcParams["svg.fonttype"] = "path"   # illustrator drops live text
plt.rcParams["pdf.fonttype"] = 42

# display name to file stem; the hyphens and the dot do not appear in any filename
NAME_TO_STEM = {"1NC9": "1NC9", "CH235.12": "CH235_12", "N49P7": "N49P7",
                "NIH45-46": "NIH4546", "VRC-PG20": "VRCPG20", "VRC01": "VRC01",
                "VRC07": "VRC07", "VRC13": "VRC13", "VRC16": "VRC16"}


def save_figure(fig, stem):
    """Write png, pdf and svg, then check the svg has no live text."""
    path = f"{fig_dir}/{stem}"
    fig.savefig(f"{path}.png", dpi=PNG_DPI, bbox_inches="tight")
    fig.savefig(f"{path}.pdf", bbox_inches="tight")
    fig.savefig(f"{path}.svg", bbox_inches="tight")
    assert open(f"{path}.svg").read().count("<text") == 0, "live text in svg, check svg.fonttype"


# ---- constants (inlined from the workflow's config.py) -----------------
ANTIBODIES = ["1NC9", "CH235.12", "VRC01", "VRC13", "VRC16",
              "VRC07", "N49P7", "NIH45-46", "VRC-PG20"]
ANTIGENS = ("CH505TF", "SF162")

GERMLINE_TOKEN = "G"        # site_seq character meaning "germline at this site"
VALUE_COL_RAW = "nlog10_Kd_raw"
FLOOR_CENSOR_VALUE = 6.0    # assay floor; values at exactly this are censored

GERMLINE = GERMLINE_TOKEN
VALUE_COL = VALUE_COL_RAW

## 3. Data loaders

In [ ]:
def load_raw_binding_data():
    """Pooled genotype table assembled from the per-antibody kds files.

    One row per antibody, antigen and genotype, replicate-averaged, with the
    G/M token string in site_seq. Reconstructs what the old pooled
    final_cleaned_all_antibody_binding_data.csv provided.
    """
    frames = []
    for ab in NAME_TO_STEM:
        d = pd.read_csv(f"{data_dir}/{NAME_TO_STEM[ab]}_kds.csv")
        d["antibody"] = ab
        d["site_seq"] = d[[f"site_{i}" for i in range(10)]].agg("".join, axis=1)
        frames.append(d)
    df = pd.concat(frames, ignore_index=True)
    df = (df.groupby(["antibody", "antigen", "site_seq", "id"], as_index=False)["nlog10_Kd"]
            .mean()
            .rename(columns={"nlog10_Kd": "nlog10_Kd_raw"}))
    print(f"Loaded raw binding data: {df.shape} from {len(NAME_TO_STEM)} kds files")
    return df


def find_metadata_file(antibody):
    """Path to one antibody's site metadata."""
    return f"{data_dir}/{NAME_TO_STEM[antibody]}_meta.csv"


def kabat_to_site_index(antibody):
    # {'L45P': 3, ...} -- maps a mutation label to its position in site_seq.
    meta = pd.read_csv(find_metadata_file(antibody))
    meta.columns = meta.columns.str.strip()
    mapping = {}
    for _, row in meta.iterrows():
        site_index = int(str(row["site"]).replace("site_", ""))
        mapping[f"{row['germline_aa']}{row['kabat_position']}{row['mature_aa']}"] = site_index
    return mapping

## 4. Shared helpers (inlined)

In [ ]:
ORDER0_COLOR = "#6B5894"  # purple -- germline
ORDER1_COLORS = ["#8EB4FB", "#002792", "#386A4F"]  # light blue, dark blue, green
ORDER2_COLORS = ["#CFB454", "#B96040", "#CC688B"]  # yellow, orange, light pink
ORDER3_COLOR = "#741345"  # magenta -- all three


def build_split_frame(df, antibody, antigen, focal_mut, contingent_mut, remaining_muts, k2i):
    f_idx = k2i[focal_mut]
    cont_idx = k2i[contingent_mut]
    rem_idx = [k2i[m] for m in remaining_muts]

    d = df[(df.antibody == antibody) & (df.antigen == antigen)].dropna(subset=[VALUE_COL]).copy()
    d["_f"] = d["site_seq"].str[f_idx] != GERMLINE

    def bgkey(seq):
        return seq[:f_idx] + seq[f_idx + 1:]
    d["_bg"] = d["site_seq"].apply(bgkey)
    d["_cont"] = d["site_seq"].str[cont_idx] != GERMLINE
    d["_rem"] = d["site_seq"].apply(lambda s: tuple(i for i in rem_idx if s[i] != GERMLINE))

    germ = d[~d["_f"]].groupby("_bg").agg(germ_val=(VALUE_COL, "mean"), cont=("_cont", "first"), rem=("_rem", "first"))
    mat = d[d["_f"]].groupby("_bg")[VALUE_COL].mean().rename("mat_val")
    merged = germ.join(mat, how="inner")
    merged["delta"] = merged["mat_val"] - merged["germ_val"]
    return merged, rem_idx

## 5. Figure code

In [ ]:
matplotlib.use("Agg")


plt.rcParams.update({"font.size": 10})

FLOOR = FLOOR_CENSOR_VALUE


def _matched_pairs(df, antibody, antigen, focal_mut, cond_mut, remaining_muts, k2i, cond_state=True):
    """cond_mut-restricted matched pairs for focal_mut (L45P), with
    remaining_muts' presence encoded so panels can filter on them.
    cond_state=True (default) restricts to cond_mut MUTANT (e.g. N52K-mutant
    only); cond_state=False restricts to cond_mut GERMLINE instead."""
    merged, rem_idx = build_split_frame(df, antibody, antigen, focal_mut, cond_mut, remaining_muts, k2i)
    merged = merged[merged["cont"] == cond_state]
    return merged, rem_idx


def _matched_pairs_by_cond(df, antibody, antigen, focal_mut, cond_mut, remaining_muts, k2i):
    """focal_mut germline-vs-mature matched pairs, NOT restricted to
    cond_mut-mutant -- both cond_mut states kept, 'cont' column (from
    build_split_frame's contingent_mut=cond_mut) used directly as the
    cond_mut-state split. This is the correct comparison for testing
    whether focal_mut's own effect is SPECIFIC to cond_mut-mutant
    backgrounds: cond_mut's large main effect is fixed within each panel
    (both germline or both mutant), so it can't swamp focal_mut's
    modulation the way a neither-vs-both(focal_mut+cond_mut) framing
    does."""
    merged, rem_idx = build_split_frame(df, antibody, antigen, focal_mut, cond_mut, remaining_muts, k2i)
    return merged, rem_idx


def _double_matched_pairs_by_cond(df, antibody, antigen, focal_muts, cond_mut, remaining_muts, k2i):
    """Matched PAIR is (NEITHER of focal_muts present) vs (BOTH present),
    for the same background -- rows with exactly one focal_mut present are
    dropped. Matching key excludes BOTH focal_muts' positions, but STILL
    includes cond_mut's and remaining_muts' positions, so each matched pair
    has a fixed, tracked cond_mut/remaining_muts state (returned as 'cont'/
    'rem' columns, same shape as _matched_pairs_by_cond) -- unlike the
    single-focal-mut version, this tests whether a JOINT double-mutation
    effect (e.g. VRC07's S54F+N52K combined benefit) is itself gated by
    cond_mut (e.g. H35N), which is a well-posed question when the pair
    IS the object of interest (contrast with pairing one focal_mut against
    a cond_mut that's already known to dominate it, which conflates the
    two main effects instead of testing gating)."""
    f_idxs = [k2i[m] for m in focal_muts]
    c_idx = k2i[cond_mut]
    rem_idx = [k2i[m] for m in remaining_muts]
    d = df[(df.antibody == antibody) & (df.antigen == antigen)].dropna(subset=[VALUE_COL_RAW]).copy()

    def bgkey(seq):
        return "".join(c for i, c in enumerate(seq) if i not in f_idxs)
    d["_bg"] = d["site_seq"].apply(bgkey)
    d["_state"] = d["site_seq"].apply(lambda s: tuple(s[i] != GERMLINE for i in f_idxs))
    d["_cond"] = d["site_seq"].str[c_idx] != GERMLINE
    d["_rem"] = d["site_seq"].apply(lambda s: tuple(i for i in rem_idx if s[i] != GERMLINE))

    neither = d[d["_state"].apply(lambda t: not any(t))].groupby("_bg").agg(
        germ_val=(VALUE_COL_RAW, "mean"), cont=("_cond", "first"), rem=("_rem", "first"))
    both = d[d["_state"].apply(all)].groupby("_bg")[VALUE_COL_RAW].mean().rename("mat_val")
    merged = neither.join(both, how="inner")
    return merged, rem_idx


def build_synergy_pair_frame(df, antibody, antigen, focal_mut, cond_mut, remaining_muts, k2i):
    """The actual SYNERGY (interaction) term, not a raw-value comparison:
    for each OUTER background (site_seq with BOTH focal_mut's and
    cond_mut's positions excluded -- so remaining_muts and every other site
    stay fixed), requires all 4 corners of {focal_mut, cond_mut} to be
    present, and returns delta_x = focal_mut's own matched-pair delta when
    cond_mut is GERMLINE, delta_y = focal_mut's own delta when cond_mut is
    MUTANT, for that same outer background. `synergy = delta_y - delta_x`
    is the difference-in-differences -- positive means cond_mut amplifies
    focal_mut's benefit (true synergy), near zero or negative means that
    amplification is absent/reversed. This is the correct quantity for
    testing whether a synergy specifically DIMINISHES as some other factor
    accumulates -- a plain matched-pair genotype scatter (raw germ_val vs
    mat_val) can't isolate this, because it's swamped by cond_mut's own
    main effect (see plot_vrc07_s54f_matched_pair_scatter.py's earlier,
    corrected attempts)."""
    f_idx = k2i[focal_mut]
    c_idx = k2i[cond_mut]
    rem_idx = [k2i[m] for m in remaining_muts]
    d = df[(df.antibody == antibody) & (df.antigen == antigen)].dropna(subset=[VALUE_COL_RAW]).copy()

    def outer_key(seq):
        return "".join(ch for i, ch in enumerate(seq) if i not in (f_idx, c_idx))
    d["_bg"] = d["site_seq"].apply(outer_key)
    d["_f"] = d["site_seq"].str[f_idx] != GERMLINE
    d["_c"] = d["site_seq"].str[c_idx] != GERMLINE
    d["_rem"] = d["site_seq"].apply(lambda s: tuple(i for i in rem_idx if s[i] != GERMLINE))

    pt = d.pivot_table(index="_bg", columns=["_f", "_c"], values=VALUE_COL_RAW, aggfunc="mean")
    for col in [(False, False), (True, False), (False, True), (True, True)]:
        if col not in pt.columns:
            pt[col] = float("nan")
    delta_x = pt[(True, False)] - pt[(False, False)]  # focal delta, cond_mut germline
    delta_y = pt[(True, True)] - pt[(False, True)]     # focal delta, cond_mut mutant
    rem = d.groupby("_bg")["_rem"].first()
    merged = pd.DataFrame({"delta_x": delta_x, "delta_y": delta_y, "rem": rem}).dropna()
    return merged, rem_idx


_JITTER_RNG = np.random.default_rng(0)


_JITTER_GAP = 0.10     # minimum clearance between the floor line and the closest jittered point
_JITTER_SPREAD = 0.10  # additional spread below that gap -- total reach is gap+spread=0.20 (floor-0.20)


def _jitter_floor(vals, floor=FLOOR, gap=_JITTER_GAP, spread=_JITTER_SPREAD):
    """Points sitting exactly AT the floor-censoring value pile up into one
    overplotted blob -- spreads them into a strip BELOW the floor line
    (never above, so they stay visually distinct from real measurements)
    via a fixed-seed uniform jitter. A visible GAP is kept between the floor
    line itself and the nearest jittered point (rather than jittering right
    up against it) so the strip reads unambiguously as "thresholded," not as
    noise sitting on the line. Non-floor values pass through unchanged."""
    vals = np.asarray(vals, dtype=float)
    at_floor = np.abs(vals - floor) < 1e-6
    out = vals.copy()
    out[at_floor] = floor - gap - _JITTER_RNG.uniform(0.0, spread, size=at_floor.sum())
    return out


def _global_range(value_arrays, floor=FLOOR, pad_frac=0.04, jitter_room=_JITTER_GAP + _JITTER_SPREAD):
    """ONE shared (lo, hi) range spanning every array passed in -- call with
    every panel's germ_vals/mat_vals from a whole figure before drawing any
    panel, so all panels in that figure share one axis range. jitter_room
    reserves enough headroom below the floor for the full jittered strip
    (gap + spread) plus a little extra padding so it's never clipped."""
    all_vals = np.concatenate([np.asarray(v, dtype=float) for v in value_arrays if len(v)])
    lo_data = min(floor, float(all_vals.min()))
    hi_data = float(all_vals.max())
    span = max(hi_data - lo_data, 1e-6)
    lo = lo_data - max(span * pad_frac, jitter_room + 0.03)
    hi = hi_data + span * pad_frac
    return lo, hi


def _relabel_floor_ticks(ax, floor=FLOOR):
    """No tick labels below the floor -- the jittered strip down there is a
    display trick, not real sub-floor data, so it shouldn't get its own
    axis numbers (e.g. '5.8'). Drop any auto-placed tick below floor and
    leave the floor tick itself BLANK (2026-08-04: was '<=floor' text;
    left blank instead so a hand-placed annotation/graphic can go there
    manually -- don't reintroduce the text label without being asked)."""
    for get_ticks, set_ticks, set_labels in (
        (ax.get_xticks, ax.set_xticks, ax.set_xticklabels),
        (ax.get_yticks, ax.set_yticks, ax.set_yticklabels),
    ):
        kept = [t for t in get_ticks() if t >= floor - 1e-9]
        labels = ["" if abs(t - floor) < 1e-6 else f"{t:g}" for t in kept]
        set_ticks(kept)
        set_labels(labels)


def _add_row_headers(fig, axes, row_labels, pad=0.015, fontsize=13, sub_label=None, sub_pad=0.014,
                      sub_fontsize=9.5):
    """One bold, centered label above each ROW of panels (spanning that
    row's columns), instead of repeating the antigen name in every panel's
    own title. Call AFTER plt.tight_layout() so axes positions are final.
    `sub_label`: optional smaller, non-bold line placed directly BELOW the
    antigen name (same one for every row, between the antigen name and the
    panels themselves) -- for a condition that's CONSTANT across the whole
    figure (e.g. "+N52K"), so it doesn't need to be repeated in every
    panel's own x/y axis label."""
    n_rows, n_cols = axes.shape
    header_pad = pad + sub_pad if sub_label else pad
    for row in range(n_rows):
        positions = [axes[row, col].get_position() for col in range(n_cols)]
        x0 = min(p.x0 for p in positions)
        x1 = max(p.x1 for p in positions)
        y1 = max(p.y1 for p in positions)
        fig.text((x0 + x1) / 2, y1 + header_pad, row_labels[row], ha="center", va="bottom",
                  fontsize=fontsize, fontweight="bold")
        if sub_label:
            fig.text((x0 + x1) / 2, y1 + pad, sub_label, ha="center", va="bottom",
                      fontsize=sub_fontsize, fontweight="normal")


def _draw_combo_key(fig, axes, combo_muts, combo_colors, combo_counts, position="right", gap=None,
                     col_width_in=0.34, cell=0.045, label_margin_in=0.42):
    """2026-08-03: dot-matrix-style key, REPLACES the old fig.legend of
    colored circle markers + text labels ("none"/"S54G"/"I51L"/
    "S54G+I51L") for every combo-colored scatter figure in this module.
    One color swatch per combo (top row, in combo_colors' own dict order --
    matches how _order_combo_palette/the hand-written _COMBO_COLORS dicts
    are already built: () | singles | pairs | triple), a dot-matrix row
    below it per constituent mutation (dark filled #222222 = that mutation
    IS part of this combo, light gray #e6e6e6 = absent -- the SAME
    present/absent encoding this project's order-violin dot-matrix panels
    already use), and an "n=" row at the bottom with the actual point
    count for that combo across every panel in the figure. The dot pattern
    IS the label now -- no more spelling out "S54G+I51L" as text.
    `position`: "right" (default, single-column 1-antigen-per-row figures
    like matched_pair_full_combo_scatter -- narrow strip just right of the
    panel grid, sized via col_width_in) or "below" (2-column, 2x2-ish grids
    like cond_selectivity_by_combo_scatter/synergy_by_combo_scatter, where
    a right-side strip was reported overlapping the panels -- instead spans
    the full width of the panel grid, sitting below the bottom row with
    enough gap to clear its x-tick labels/xlabel text). The swatch/dot-
    matrix area is inset from the key axes' own left edge by
    `label_margin_in` (a fixed physical width, converted to a fraction of
    THIS key's own width below) so the mutation-name row labels have a
    reserved gutter to sit in -- for `position="below"` that gutter is
    carved out of the panel grid's own width rather than extending past
    it, so labels never bleed into the y-axis label area of the leftmost
    panel. `col_width_in`: physical width PER COLUMN in inches for
    `position="right"` (needs to scale with figure size, not a fixed
    figure-fraction, since a fixed fraction left 3-digit n= counts
    overlapping on narrow 1-panel figures while being oversized on wide
    ones); ignored for `position="below"`, which instead fills the panel
    grid's own width. Call AFTER plt.tight_layout() so axes positions
    (used to place the key relative to the panel grid) are final."""
    combos = list(combo_colors.keys())
    ncol = len(combos)
    nrow = len(combo_muts)

    fig_w_in, fig_h_in = fig.get_size_inches()
    x0 = min(ax.get_position().x0 for ax in np.asarray(axes).flat)
    x1 = max(ax.get_position().x1 for ax in np.asarray(axes).flat)
    y0 = min(ax.get_position().y0 for ax in np.asarray(axes).flat)
    y1 = max(ax.get_position().y1 for ax in np.asarray(axes).flat)

    if position == "below":
        # y0 above is just the DATA area's bottom edge -- tick labels and
        # the "-{focal_mut} affinity" xlabel extend further down than that.
        # get_tightbbox includes them, so use ITS bottom edge as the real
        # floor to place the key below (fixed constants here badly
        # underestimated that extent and the key ended up overlapping the
        # xlabel text).
        fig.canvas.draw()
        renderer = fig.canvas.get_renderer()
        tight_y0 = min(
            ax.get_tightbbox(renderer).transformed(fig.transFigure.inverted()).y0
            for ax in np.asarray(axes).flat
        )
        gap = 0.015 if gap is None else gap
        height = (nrow + 2) * cell
        key_w = x1 - x0
        y0 = tight_y0  # anchor the key relative to the true (label-inclusive) bottom edge
        key_ax = fig.add_axes([x0, y0 - gap - height, key_w, height])
    else:
        gap = 0.03 if gap is None else gap
        height = min((nrow + 2) * cell, y1 - y0)
        key_w = col_width_in * ncol / fig_w_in
        key_ax = fig.add_axes([x1 + gap, (y0 + y1) / 2 - height / 2, key_w, height])

    # normalized 0..1 x-axis: [0, label_frac) reserved for row labels (never
    # drawn on), [label_frac, 1] holds the ncol swatch/dot-matrix columns
    label_frac = min(0.35, (label_margin_in / fig_w_in) / key_w)
    col_w = (1 - label_frac) / ncol
    key_ax.set_xlim(0, 1)
    key_ax.set_ylim(-1.3, nrow + 1.0)
    key_ax.axis("off")

    def col_x(i):
        return label_frac + (i + 0.5) * col_w

    for i, combo in enumerate(combos):
        key_ax.add_patch(plt.Rectangle((label_frac + i * col_w + 0.06 * col_w, nrow + 0.15),
                                        0.88 * col_w, 0.55, facecolor=combo_colors[combo],
                                        edgecolor="none", zorder=3))

    row_ys = list(range(nrow - 1, -1, -1))
    for mut, ry in zip(combo_muts, row_ys):
        key_ax.text(label_frac - 0.02, ry + 0.5, mut, ha="right", va="center", fontsize=8.5)
        for i, combo in enumerate(combos):
            present = mut in combo
            key_ax.scatter(col_x(i), ry + 0.5, s=34, color="#222222" if present else "#e6e6e6",
                            edgecolor="none", zorder=4)

    key_ax.text(label_frac - 0.02, -0.7, "n=", ha="right", va="center", fontsize=8.5, color="#262626")
    for i, combo in enumerate(combos):
        key_ax.text(col_x(i), -0.7, str(int(combo_counts.get(combo, 0))), ha="center", va="center",
                    fontsize=8.5, color="#262626")


def _scatter_panel(ax, germ_vals, mat_vals, color, title, lo, hi, n_label=True, xlabel=None, ylabel=None):
    germ_vals = np.asarray(germ_vals, dtype=float)
    mat_vals = np.asarray(mat_vals, dtype=float)
    germ_plot = _jitter_floor(germ_vals)
    mat_plot = _jitter_floor(mat_vals)

    ax.plot([lo, hi], [lo, hi], color="#999999", linewidth=1, linestyle="--", zorder=1)
    ax.axvline(FLOOR, color="#d9534f", linewidth=0.8, linestyle=":", zorder=1, alpha=0.7)
    ax.axhline(FLOOR, color="#d9534f", linewidth=0.8, linestyle=":", zorder=1, alpha=0.7)
    ax.scatter(germ_plot, mat_plot, s=16, color=color, alpha=.85,
               edgecolor="none", zorder=3)
    ax.set_xlim(lo, hi)
    ax.set_ylim(lo, hi)
    ax.set_aspect("equal")
    _relabel_floor_ticks(ax)
    n = len(germ_vals)
    label = f"{title} (n={n})" if n_label else title
    ax.set_title(label, fontsize=10, fontweight="normal")
    if xlabel:
        ax.set_xlabel(xlabel, fontsize=9)
    if ylabel:
        ax.set_ylabel(ylabel, fontsize=9)
    for sp in ["top", "right"]:
        ax.spines[sp].set_visible(False)


def matched_pair_scatter_grid(df, antibody, focal_mut, cond_mut, panel_states, out_name,
                               suptitle, remaining_muts, colors=None, restrict_germline_at=None,
                               cond_state=True):
    """panel_states: list of (label, filter_fn(rem_tuple, rem_idx_lookup) -> bool).
    remaining_muts: the site(s) panel_states filters on (passed to build_split_frame
    as remaining_muts so their presence is tracked in each matched pair's 'rem').
    cond_state: True (default) = cond_mut MUTANT only; False = cond_mut GERMLINE only."""
    k2i = kabat_to_site_index(antibody)
    n_panels = len(panel_states)
    colors = colors or ([ORDER1_COLORS[0]] * n_panels)
    cond_sign = "+" if cond_state else "-"

    # first pass: gather every panel's data so the axis range can be global
    panel_data = {}
    all_vals = []
    for row, antigen in enumerate(ANTIGENS):
        d = df[(df.antibody == antibody) & (df.antigen == antigen)].dropna(subset=[VALUE_COL_RAW]).copy()
        for m in (restrict_germline_at or []):
            gi = k2i[m]
            d = d[d["site_seq"].str[gi] == GERMLINE]
        merged, rem_idx = _matched_pairs(d, antibody, antigen, focal_mut, cond_mut, remaining_muts, k2i,
                                          cond_state=cond_state)
        for col, (label, filt) in enumerate(panel_states):
            sub = merged[merged["rem"].apply(lambda r: filt(r, rem_idx))]
            gv, mv = sub["germ_val"].values, sub["mat_val"].values
            panel_data[(row, col)] = (gv, mv, label)
            all_vals.append(gv)
            all_vals.append(mv)
    lo, hi = _global_range(all_vals)

    fig, axes = plt.subplots(len(ANTIGENS), n_panels,
                              figsize=(3.1 * n_panels, 3.4 * len(ANTIGENS)), dpi=170,
                              squeeze=False)
    for row in range(len(ANTIGENS)):
        for col in range(n_panels):
            gv, mv, label = panel_data[(row, col)]
            ax = axes[row, col]
            _scatter_panel(ax, gv, mv, colors[col], label, lo, hi)
            if col == 0:
                ax.set_ylabel(f"+{focal_mut} affinity", fontsize=9.5)
            if row == len(ANTIGENS) - 1:
                ax.set_xlabel(f"-{focal_mut} affinity", fontsize=9.5)

    plt.tight_layout()
    # cond_mut's state (e.g. "+N52K") is CONSTANT across this whole figure
    # (that's what cond_state pins) -- shown once per antigen row header
    # instead of repeated in every panel's own x/y axis label.
    _add_row_headers(fig, axes, list(ANTIGENS), sub_label=f"{cond_sign}{cond_mut}")
    fig.suptitle(suptitle, fontsize=12.5, fontweight="bold", y=1.08)

    save_figure(fig, Path(out_name).stem)
    plt.close(fig)


def cond_selectivity_scatter_grid(df, antibody, focal_mut, cond_mut, panel_states, out_name, suptitle,
                                   remaining_muts, colors=None, restrict_germline_at=None, restrict_mutant_at=None):
    """focal_mut germline-vs-mature scatter, panels = cond_mut state (x other
    split dims), antigens as rows -- the direct visual test of 'is
    focal_mut's effect specific to cond_mut-mutant backgrounds' with
    cond_mut's own main effect held fixed within each panel (see
    _matched_pairs_by_cond). restrict_germline_at/restrict_mutant_at pin
    OTHER sites (not focal_mut/cond_mut) to a fixed state throughout, e.g.
    VRC07's H35N-gated S54F:N52K story needs H35N pinned mutant."""
    k2i = kabat_to_site_index(antibody)
    n_panels = len(panel_states)
    colors = colors or ([ORDER1_COLORS[0]] * n_panels)

    panel_data = {}
    all_vals = []
    for row, antigen in enumerate(ANTIGENS):
        d = df[(df.antibody == antibody) & (df.antigen == antigen)].dropna(subset=[VALUE_COL_RAW]).copy()
        for m in (restrict_germline_at or []):
            d = d[d["site_seq"].str[k2i[m]] == GERMLINE]
        for m in (restrict_mutant_at or []):
            d = d[d["site_seq"].str[k2i[m]] != GERMLINE]
        merged, rem_idx = _matched_pairs_by_cond(d, antibody, antigen, focal_mut, cond_mut, remaining_muts, k2i)
        for col, (label, filt) in enumerate(panel_states):
            sub = merged[merged.apply(lambda row_: filt(row_["cont"], row_["rem"], rem_idx), axis=1)]
            gv, mv = sub["germ_val"].values, sub["mat_val"].values
            panel_data[(row, col)] = (gv, mv, label)
            all_vals.append(gv)
            all_vals.append(mv)
    lo, hi = _global_range(all_vals)

    fig, axes = plt.subplots(len(ANTIGENS), n_panels,
                              figsize=(3.1 * n_panels, 3.4 * len(ANTIGENS)), dpi=170,
                              squeeze=False)
    for row in range(len(ANTIGENS)):
        for col in range(n_panels):
            gv, mv, label = panel_data[(row, col)]
            ax = axes[row, col]
            _scatter_panel(ax, gv, mv, colors[col], label, lo, hi)
            if col == 0:
                ax.set_ylabel(f"+{focal_mut} affinity", fontsize=9.5)
            if row == len(ANTIGENS) - 1:
                ax.set_xlabel(f"-{focal_mut} affinity", fontsize=9.5)

    plt.tight_layout()
    _add_row_headers(fig, axes, list(ANTIGENS))
    fig.suptitle(suptitle, fontsize=12.5, fontweight="bold", y=1.08)

    save_figure(fig, Path(out_name).stem)
    plt.close(fig)


def _scatter_panel_multicolor(ax, germ_vals, mat_vals, point_colors, title, lo, hi, xlabel=None, ylabel=None):
    germ_vals = np.asarray(germ_vals, dtype=float)
    mat_vals = np.asarray(mat_vals, dtype=float)
    germ_plot = _jitter_floor(germ_vals)
    mat_plot = _jitter_floor(mat_vals)

    ax.plot([lo, hi], [lo, hi], color="#999999", linewidth=1, linestyle="--", zorder=1)
    ax.axvline(FLOOR, color="#d9534f", linewidth=0.8, linestyle=":", zorder=1, alpha=0.7)
    ax.axhline(FLOOR, color="#d9534f", linewidth=0.8, linestyle=":", zorder=1, alpha=0.7)
    ax.scatter(germ_plot, mat_plot, s=16, color=point_colors, alpha=.85,
               edgecolor="none", zorder=3)
    ax.set_xlim(lo, hi)
    ax.set_ylim(lo, hi)
    ax.set_aspect("equal")
    _relabel_floor_ticks(ax)
    ax.set_title(f"{title} (n={len(germ_vals)})", fontsize=10, fontweight="normal")
    if xlabel:
        ax.set_xlabel(xlabel, fontsize=9)
    if ylabel:
        ax.set_ylabel(ylabel, fontsize=9)
    for sp in ["top", "right"]:
        ax.spines[sp].set_visible(False)


# count-of-background-mutations palette -- reuses this project's established
# germline/single/double/triple ORDER0-3 color family (three_way_order_violin
# convention) rather than inventing a new count palette, since "count of a
# background set" IS exactly that germline/single/double/triple concept.
_COUNT_COLORS = [ORDER0_COLOR, ORDER1_COLORS[0], ORDER2_COLORS[0], ORDER3_COLOR]

# specific-combo categorical palette for the S54G x I51L identity breakdown
# (muted/desaturated qualitative set, distinct from the count palette above
# since these encode IDENTITY not magnitude): dark navy / teal / mustard /
# dusty mauve.
_COMBO_COLORS = {
    (): "#2b3a67",
    ("S54G",): "#3f8073",
    ("I51L",): "#c9a227",
    ("S54G", "I51L"): "#a8577e",
}
_COMBO_LABELS = {(): "none", ("S54G",): "S54G", ("I51L",): "I51L", ("S54G", "I51L"): "S54G+I51L"}


def cond_selectivity_by_bg_count_scatter(df, antibody, focal_mut, cond_mut, bg_muts, out_name, suptitle,
                                          antigens=None, restrict_germline_at=None, restrict_mutant_at=None,
                                          count_colors=None):
    """focal_mut germline-vs-mature scatter, 2 panels (cond_mut germline /
    mutant), ALL points overlaid in one panel per state and colored by
    COUNT of bg_muts present (0..len(bg_muts)) -- the direct analogue of
    the K19T/S56R+T57P reference figure's '# of {N52D,S53R,G54R}' styling."""
    k2i = kabat_to_site_index(antibody)
    antigens = antigens or list(ANTIGENS)
    panel_states = [(f"-{cond_mut}", False), (f"+{cond_mut}", True)]
    count_colors = count_colors or _COUNT_COLORS

    panel_data = {}
    all_vals = []
    for row, antigen in enumerate(antigens):
        d = df[(df.antibody == antibody) & (df.antigen == antigen)].dropna(subset=[VALUE_COL_RAW]).copy()
        for m in (restrict_germline_at or []):
            d = d[d["site_seq"].str[k2i[m]] == GERMLINE]
        for m in (restrict_mutant_at or []):
            d = d[d["site_seq"].str[k2i[m]] != GERMLINE]
        merged, rem_idx = _matched_pairs_by_cond(d, antibody, antigen, focal_mut, cond_mut, bg_muts, k2i)
        for col, (label, cont_state) in enumerate(panel_states):
            sub = merged[merged["cont"] == cont_state]
            counts = sub["rem"].apply(len)
            panel_data[(row, col)] = (sub["germ_val"].values, sub["mat_val"].values, counts, label)
            all_vals.append(sub["germ_val"].values)
            all_vals.append(sub["mat_val"].values)
    lo, hi = _global_range(all_vals)

    fig, axes = plt.subplots(len(antigens), 2, figsize=(7.2, 3.6 * len(antigens)), dpi=170, squeeze=False)
    legend_handles = None
    for row in range(len(antigens)):
        for col, (label, cont_state) in enumerate(panel_states):
            gv, mv, counts, _ = panel_data[(row, col)]
            ax = axes[row, col]
            point_colors = [count_colors[c] for c in counts]
            _scatter_panel_multicolor(
                ax, gv, mv, point_colors, label, lo, hi,
                xlabel=f"-{focal_mut} affinity" if row == len(antigens) - 1 else None,
                ylabel=f"+{focal_mut} affinity" if col == 0 else None,
            )
            if legend_handles is None and col == 1:
                ns = counts.value_counts().reindex(range(len(bg_muts) + 1), fill_value=0)
                legend_handles = [
                    Line2D([0], [0], marker="o", linestyle="", markerfacecolor=count_colors[c],
                           markeredgecolor="black", markersize=7,
                           label=f"{c} of {len(bg_muts)} (n={ns.get(c, 0)})")
                    for c in range(len(bg_muts) + 1)
                ]

    plt.tight_layout()
    _add_row_headers(fig, axes, antigens)
    fig.legend(handles=legend_handles, title=f"# of {{{','.join(bg_muts)}}}",
               loc="center left", bbox_to_anchor=(1.0, 0.5), fontsize=8.5, title_fontsize=9, frameon=False)
    fig.suptitle(suptitle, fontsize=12.5, fontweight="bold", y=1.1)

    save_figure(fig, Path(out_name).stem)
    plt.close(fig)


def cond_selectivity_by_combo_scatter(df, antibody, focal_mut, cond_mut, combo_muts, out_name, suptitle,
                                       antigens=None, restrict_germline_at=None, restrict_mutant_at=None,
                                       combo_colors=None, combo_labels=None, show_title=True):
    """Same layout as cond_selectivity_by_bg_count_scatter, but colored by
    the SPECIFIC identity combo of combo_muts (not just a count) -- the
    analogue of the second reference figure's named-combo categorical
    legend/palette, with floor-jittered points."""
    k2i = kabat_to_site_index(antibody)
    antigens = antigens or list(ANTIGENS)
    panel_states = [(f"-{cond_mut}", False), (f"+{cond_mut}", True)]
    combo_colors = combo_colors or _COMBO_COLORS
    combo_labels = combo_labels or _COMBO_LABELS
    idx_by_name = {m: k2i[m] for m in combo_muts}

    def combo_of(rem_tuple, rem_idx):
        return tuple(m for m in combo_muts if idx_by_name[m] in rem_tuple)

    panel_data = {}
    all_vals = []
    for row, antigen in enumerate(antigens):
        d = df[(df.antibody == antibody) & (df.antigen == antigen)].dropna(subset=[VALUE_COL_RAW]).copy()
        for m in (restrict_germline_at or []):
            d = d[d["site_seq"].str[k2i[m]] == GERMLINE]
        for m in (restrict_mutant_at or []):
            d = d[d["site_seq"].str[k2i[m]] != GERMLINE]
        merged, rem_idx = _matched_pairs_by_cond(d, antibody, antigen, focal_mut, cond_mut, combo_muts, k2i)
        for col, (label, cont_state) in enumerate(panel_states):
            sub = merged[merged["cont"] == cont_state]
            combos = sub["rem"].apply(lambda r: combo_of(r, rem_idx))
            panel_data[(row, col)] = (sub["germ_val"].values, sub["mat_val"].values, combos, label)
            all_vals.append(sub["germ_val"].values)
            all_vals.append(sub["mat_val"].values)
    lo, hi = _global_range(all_vals)

    fig, axes = plt.subplots(len(antigens), 2, figsize=(7.6, 3.6 * len(antigens)), dpi=170, squeeze=False)
    for row in range(len(antigens)):
        for col, (label, cont_state) in enumerate(panel_states):
            gv, mv, combos, _ = panel_data[(row, col)]
            ax = axes[row, col]
            point_colors = [combo_colors[c] for c in combos]
            _scatter_panel_multicolor(
                ax, gv, mv, point_colors, label, lo, hi,
                xlabel=f"-{focal_mut} affinity" if row == len(antigens) - 1 else None,
                ylabel=f"+{focal_mut} affinity" if col == 0 else None,
            )

    plt.tight_layout()
    _add_row_headers(fig, axes, antigens)
    combo_counts = pd.concat([panel_data[k][2] for k in panel_data]).value_counts()
    _draw_combo_key(fig, axes, combo_muts, combo_colors, combo_counts, position="below")
    if show_title:
        fig.suptitle(suptitle, fontsize=12.5, fontweight="bold", y=1.1)

    save_figure(fig, Path(out_name).stem)
    plt.close(fig)


def double_matched_by_cond_combo_scatter(df, antibody, focal_muts, cond_mut, combo_muts, out_name, suptitle,
                                          antigens=None, combo_colors=None, combo_labels=None):
    """The JOINT double-mutation counterpart to cond_selectivity_by_combo_scatter:
    x = NEITHER of focal_muts present, y = BOTH present (see
    _double_matched_pairs_by_cond), 2 panels = cond_mut state, colored by
    the specific identity combo of combo_muts. Use this when the PAIR
    itself (not one focal_mut modulated by a dominant cond_mut) is the
    thing being tested for cond_mut-gating -- e.g. is VRC07's combined
    S54F+N52K benefit itself gated by H35N."""
    k2i = kabat_to_site_index(antibody)
    antigens = antigens or list(ANTIGENS)
    panel_states = [(f"-{cond_mut}", False), (f"+{cond_mut}", True)]
    combo_colors = combo_colors or _COMBO_COLORS
    combo_labels = combo_labels or _COMBO_LABELS
    idx_by_name = {m: k2i[m] for m in combo_muts}

    def combo_of(rem_tuple):
        return tuple(m for m in combo_muts if idx_by_name[m] in rem_tuple)

    panel_data = {}
    all_vals = []
    for row, antigen in enumerate(antigens):
        merged, rem_idx = _double_matched_pairs_by_cond(df, antibody, antigen, focal_muts, cond_mut,
                                                          combo_muts, k2i)
        for col, (label, cont_state) in enumerate(panel_states):
            sub = merged[merged["cont"] == cont_state]
            combos = sub["rem"].apply(combo_of)
            panel_data[(row, col)] = (sub["germ_val"].values, sub["mat_val"].values, combos, label)
            all_vals.append(sub["germ_val"].values)
            all_vals.append(sub["mat_val"].values)
    lo, hi = _global_range(all_vals)

    # each focal_mut gets its OWN sign prefix ("-S54F -N52K", not the
    # ambiguous "-S54F+N52K" a single shared join-then-prefix would produce)
    germ_label = " ".join(f"-{m}" for m in focal_muts)
    mat_label = " ".join(f"+{m}" for m in focal_muts)
    fig, axes = plt.subplots(len(antigens), 2, figsize=(7.6, 3.6 * len(antigens)), dpi=170, squeeze=False)
    for row in range(len(antigens)):
        for col, (label, cont_state) in enumerate(panel_states):
            gv, mv, combos, _ = panel_data[(row, col)]
            ax = axes[row, col]
            point_colors = [combo_colors[c] for c in combos]
            _scatter_panel_multicolor(
                ax, gv, mv, point_colors, label, lo, hi,
                xlabel=f"{germ_label} affinity" if row == len(antigens) - 1 else None,
                ylabel=f"{mat_label} affinity" if col == 0 else None,
            )

    plt.tight_layout()
    _add_row_headers(fig, axes, antigens)
    legend_handles = [
        Line2D([0], [0], marker="o", linestyle="", markerfacecolor=combo_colors[c],
               markeredgecolor="black", markersize=7, label=combo_labels[c])
        for c in combo_colors
    ]
    fig.legend(handles=legend_handles, loc="center left", bbox_to_anchor=(1.0, 0.5),
               fontsize=8.5, frameon=False)
    fig.suptitle(suptitle, fontsize=12.5, fontweight="bold", y=1.1)

    save_figure(fig, Path(out_name).stem)
    plt.close(fig)


def _synergy_scatter_panel(ax, delta_x, delta_y, point_colors, title, lo, hi, xlabel=None, ylabel=None):
    """delta_x/delta_y are DELTAS (focal_mut's own matched-pair delta,
    cond_mut germline vs mutant), not raw affinities -- no floor-jitter
    (deltas aren't floor-censored the way raw KD is), but a zero reference
    line on both axes plus the y=x diagonal: above the diagonal = cond_mut
    amplifies focal_mut's effect (synergy), on/below = that amplification
    is absent or reversed."""
    ax.axhline(0, color="#bbbbbb", linewidth=0.8, zorder=1)
    ax.axvline(0, color="#bbbbbb", linewidth=0.8, zorder=1)
    ax.plot([lo, hi], [lo, hi], color="#999999", linewidth=1, linestyle="--", zorder=1)
    ax.scatter(delta_x, delta_y, s=16, color=point_colors, alpha=.85,
               edgecolor="none", zorder=3)
    ax.set_xlim(lo, hi)
    ax.set_ylim(lo, hi)
    ax.set_aspect("equal")
    ax.set_title(f"{title} (n={len(delta_x)})", fontsize=10, fontweight="normal")
    if xlabel:
        ax.set_xlabel(xlabel, fontsize=9)
    if ylabel:
        ax.set_ylabel(ylabel, fontsize=9)
    for sp in ["top", "right"]:
        ax.spines[sp].set_visible(False)


def synergy_by_combo_scatter(df, antibody, focal_mut, cond_mut, panel_mut, combo_muts, out_name, suptitle,
                              antigens=None, combo_colors=None, combo_labels=None,
                              restrict_germline_at=None, restrict_mutant_at=None, show_title=True):
    """The DIRECT visual test of 'does the focal_mut:cond_mut synergy
    diminish/cancel as combo_muts accumulate': x = focal_mut's delta with
    cond_mut germline, y = focal_mut's delta with cond_mut mutant (see
    build_synergy_pair_frame), 2 panels = panel_mut state (e.g. H35N -- the
    established hub/gate), colored by combo_muts identity. A point above
    the diagonal has real cond_mut-driven synergy; a point on/below it
    shows that synergy has been cancelled for that background."""
    k2i = kabat_to_site_index(antibody)
    antigens = antigens or list(ANTIGENS)
    panel_states = [(f"-{panel_mut}", False), (f"+{panel_mut}", True)]
    combo_colors = combo_colors or _COMBO_COLORS
    combo_labels = combo_labels or _COMBO_LABELS
    idx_by_name = {m: k2i[m] for m in combo_muts}

    def combo_of(rem_tuple):
        return tuple(m for m in combo_muts if idx_by_name[m] in rem_tuple)

    panel_data = {}
    all_vals = []
    for row, antigen in enumerate(antigens):
        d = df[(df.antibody == antibody) & (df.antigen == antigen)].dropna(subset=[VALUE_COL_RAW]).copy()
        for m in (restrict_germline_at or []):
            d = d[d["site_seq"].str[k2i[m]] == GERMLINE]
        for m in (restrict_mutant_at or []):
            d = d[d["site_seq"].str[k2i[m]] != GERMLINE]
        p_idx = k2i[panel_mut]
        for col, (label, panel_state) in enumerate(panel_states):
            sub_d = d[(d["site_seq"].str[p_idx] != GERMLINE) == panel_state]
            merged, rem_idx = build_synergy_pair_frame(sub_d, antibody, antigen, focal_mut, cond_mut,
                                                        combo_muts, k2i)
            combos = merged["rem"].apply(combo_of)
            panel_data[(row, col)] = (merged["delta_x"].values, merged["delta_y"].values, combos, label)
            all_vals.append(merged["delta_x"].values)
            all_vals.append(merged["delta_y"].values)

    all_concat = np.concatenate([v for v in all_vals if len(v)])
    lo, hi = float(all_concat.min()), float(all_concat.max())
    pad = max((hi - lo) * 0.08, 0.05)
    lo, hi = lo - pad, hi + pad

    fig, axes = plt.subplots(len(antigens), 2, figsize=(7.6, 3.6 * len(antigens)), dpi=170, squeeze=False)
    for row in range(len(antigens)):
        for col, (label, panel_state) in enumerate(panel_states):
            dx, dy, combos, _ = panel_data[(row, col)]
            ax = axes[row, col]
            point_colors = [combo_colors[c] for c in combos]
            _synergy_scatter_panel(
                ax, dx, dy, point_colors, label, lo, hi,
                xlabel=f"{focal_mut} delta, -{cond_mut}" if row == len(antigens) - 1 else None,
                ylabel=f"{focal_mut} delta, +{cond_mut}" if col == 0 else None,
            )

    plt.tight_layout()
    _add_row_headers(fig, axes, antigens)
    combo_counts = pd.concat([panel_data[k][2] for k in panel_data]).value_counts()
    _draw_combo_key(fig, axes, combo_muts, combo_colors, combo_counts, position="below")
    if show_title:
        fig.suptitle(suptitle, fontsize=12.5, fontweight="bold", y=1.1)

    save_figure(fig, Path(out_name).stem)
    plt.close(fig)


def synergy_scatter_simple(df, antibody, focal_mut, cond_mut, combo_muts, out_name, suptitle, antigens=None,
                            restrict_germline_at=None, restrict_mutant_at=None, combo_colors=None,
                            combo_labels=None):
    """Same idea as synergy_by_combo_scatter (x = focal_mut's delta with
    cond_mut germline, y = focal_mut's delta with cond_mut mutant -- see
    build_synergy_pair_frame), but NO extra panel-mut split -- just one
    panel per antigen (rows), colored by combo_muts. Use this when there's
    no third 'hub' mutation to gate on (unlike VRC07's H35N), so the
    simplest honest figure is one point cloud per antigen colored by the
    modulator(s) of interest."""
    k2i = kabat_to_site_index(antibody)
    antigens = antigens or list(ANTIGENS)
    default_colors, default_labels = _order_combo_palette(combo_muts)
    combo_colors = combo_colors or default_colors
    combo_labels = combo_labels or default_labels
    idx_by_name = {m: k2i[m] for m in combo_muts}

    def combo_of(rem_tuple):
        return tuple(m for m in combo_muts if idx_by_name[m] in rem_tuple)

    panel_data = {}
    all_vals = []
    for row, antigen in enumerate(antigens):
        d = df[(df.antibody == antibody) & (df.antigen == antigen)].dropna(subset=[VALUE_COL_RAW]).copy()
        for m in (restrict_germline_at or []):
            d = d[d["site_seq"].str[k2i[m]] == GERMLINE]
        for m in (restrict_mutant_at or []):
            d = d[d["site_seq"].str[k2i[m]] != GERMLINE]
        merged, rem_idx = build_synergy_pair_frame(d, antibody, antigen, focal_mut, cond_mut, combo_muts, k2i)
        combos = merged["rem"].apply(combo_of)
        panel_data[row] = (merged["delta_x"].values, merged["delta_y"].values, combos)
        all_vals.append(merged["delta_x"].values)
        all_vals.append(merged["delta_y"].values)

    all_concat = np.concatenate([v for v in all_vals if len(v)])
    lo, hi = float(all_concat.min()), float(all_concat.max())
    pad = max((hi - lo) * 0.08, 0.05)
    lo, hi = lo - pad, hi + pad

    fig, axes = plt.subplots(len(antigens), 1, figsize=(4.4, 3.8 * len(antigens)), dpi=170, squeeze=False)
    for row, antigen in enumerate(antigens):
        dx, dy, combos = panel_data[row]
        ax = axes[row, 0]
        point_colors = [combo_colors[c] for c in combos]
        _synergy_scatter_panel(
            ax, dx, dy, point_colors, antigen, lo, hi,
            xlabel=f"{focal_mut} delta, -{cond_mut}" if row == len(antigens) - 1 else None,
            ylabel=f"{focal_mut} delta, +{cond_mut}",
        )

    plt.tight_layout()
    legend_handles = [
        Line2D([0], [0], marker="o", linestyle="", markerfacecolor=combo_colors[c],
               markeredgecolor="black", markersize=7, label=combo_labels[c])
        for c in combo_colors
    ]
    fig.legend(handles=legend_handles, loc="center left", bbox_to_anchor=(1.0, 0.5),
               fontsize=8.5, frameon=False)
    fig.suptitle(suptitle, fontsize=12.5, fontweight="bold", y=1.03)

    save_figure(fig, Path(out_name).stem)
    plt.close(fig)


def overlay_antigens_scatter(df, antibody, focal_mut, cond_mut, marker_mut, out_name, suptitle,
                              restrict_mutant_at=None, antigen_colors=None):
    """Both antigens OVERLAID in one panel (not stacked rows) -- color =
    antigen, marker shape = marker_mut state (circle=-marker_mut,
    triangle=+marker_mut), restricted to cond_mut-mutant throughout. Puts
    the antigen-flip itself directly in view in a single glance instead of
    requiring a top/bottom comparison across two separate panels."""
    k2i = kabat_to_site_index(antibody)
    antigens = list(ANTIGENS)
    antigen_colors = antigen_colors or {antigens[0]: ORDER1_COLORS[1], antigens[1]: ORDER2_COLORS[1]}
    marker_idx = k2i[marker_mut]

    data = {}
    all_vals = []
    for antigen in antigens:
        d = df[(df.antibody == antibody) & (df.antigen == antigen)].dropna(subset=[VALUE_COL_RAW]).copy()
        for m in (restrict_mutant_at or []):
            d = d[d["site_seq"].str[k2i[m]] != GERMLINE]
        merged, rem_idx = build_split_frame(d, antibody, antigen, focal_mut, cond_mut, [marker_mut], k2i)
        merged = merged[merged["cont"]]  # cond_mut mutant only
        data[antigen] = merged
        all_vals.append(merged["germ_val"].values)
        all_vals.append(merged["mat_val"].values)
    lo, hi = _global_range(all_vals)

    fig, ax = plt.subplots(figsize=(5.2, 5.2), dpi=170)
    ax.plot([lo, hi], [lo, hi], color="#999999", linewidth=1, linestyle="--", zorder=1)
    ax.axvline(FLOOR, color="#d9534f", linewidth=0.8, linestyle=":", zorder=1, alpha=0.7)
    ax.axhline(FLOOR, color="#d9534f", linewidth=0.8, linestyle=":", zorder=1, alpha=0.7)
    for antigen in antigens:
        merged = data[antigen]
        has_marker = merged["rem"].apply(lambda r: marker_idx in r)
        color = antigen_colors[antigen]
        for state, marker, size, alpha, label in [
            (False, "o", 30, 0.6, f"{antigen}, -{marker_mut}"),
            (True, "^", 48, 0.9, f"{antigen}, +{marker_mut}"),
        ]:
            sub = merged[has_marker == state]
            gv = _jitter_floor(sub["germ_val"].values)
            mv = _jitter_floor(sub["mat_val"].values)
            ax.scatter(gv, mv, s=size, color=color, marker=marker, alpha=alpha,
                       edgecolor="black", linewidths=0.5, zorder=3, label=label)
    ax.set_xlim(lo, hi)
    ax.set_ylim(lo, hi)
    ax.set_aspect("equal")
    _relabel_floor_ticks(ax)
    ax.set_xlabel(f"-{focal_mut} affinity", fontsize=9.5)
    ax.set_ylabel(f"+{focal_mut} affinity", fontsize=9.5)
    for sp in ["top", "right"]:
        ax.spines[sp].set_visible(False)
    ax.legend(fontsize=8, loc="upper left", frameon=False)
    fig.suptitle(suptitle, fontsize=12, fontweight="bold")
    plt.tight_layout()

    save_figure(fig, Path(out_name).stem)
    plt.close(fig)


def trend_line_combo_scatter(df, antibody, focal_mut, cond_mut, combo_muts, out_name, suptitle,
                              restrict_germline_at=None, combo_colors=None, combo_labels=None):
    """Raw matched-pair scatter (cond_mut held mutant), colored by the FULL
    combo of combo_muts (1-3 muts, up to 8 states -- reuses the
    germline/single/double/triple ORDER-family palette via
    _order_combo_palette), with a per-group linear fit line overlaid -- the
    fit lines' different slope/offset from the diagonal make a noisy
    per-point divergence visually legible without switching to a delta
    plot. Fit is computed on the RAW (non-jittered) values; only the
    plotted points are jittered for display."""
    k2i = kabat_to_site_index(antibody)
    default_colors, default_labels = _order_combo_palette(combo_muts)
    combo_colors = combo_colors or default_colors
    combo_labels = combo_labels or default_labels
    idx_by_name = {m: k2i[m] for m in combo_muts}
    antigens = list(ANTIGENS)

    def combo_of(rem_tuple):
        return tuple(m for m in combo_muts if idx_by_name[m] in rem_tuple)

    panel_data = {}
    all_vals = []
    for row, antigen in enumerate(antigens):
        d = df[(df.antibody == antibody) & (df.antigen == antigen)].dropna(subset=[VALUE_COL_RAW]).copy()
        for m in (restrict_germline_at or []):
            d = d[d["site_seq"].str[k2i[m]] == GERMLINE]
        merged, rem_idx = _matched_pairs(d, antibody, antigen, focal_mut, cond_mut, combo_muts, k2i, cond_state=True)
        combos = merged["rem"].apply(combo_of)
        panel_data[row] = (merged, combos)
        all_vals.append(merged["germ_val"].values)
        all_vals.append(merged["mat_val"].values)
    lo, hi = _global_range(all_vals)

    fig, axes = plt.subplots(len(antigens), 1, figsize=(4.6, 4.0 * len(antigens)), dpi=170, squeeze=False)
    for row, antigen in enumerate(antigens):
        merged, combos = panel_data[row]
        ax = axes[row, 0]
        ax.plot([lo, hi], [lo, hi], color="#999999", linewidth=1, linestyle="--", zorder=1)
        ax.axvline(FLOOR, color="#d9534f", linewidth=0.8, linestyle=":", zorder=1, alpha=0.7)
        ax.axhline(FLOOR, color="#d9534f", linewidth=0.8, linestyle=":", zorder=1, alpha=0.7)
        for combo, color in combo_colors.items():
            sub = merged[combos == combo]
            gv, mv = sub["germ_val"].values, sub["mat_val"].values
            if len(gv) == 0:
                continue
            ax.scatter(_jitter_floor(gv), _jitter_floor(mv), s=26, color=color, alpha=0.7,
                       edgecolor="black", linewidths=0.4, zorder=3, label=f"{combo_labels[combo]} (n={len(gv)})")
            if len(gv) >= 2:
                slope, intercept = np.polyfit(gv, mv, 1)
                xs = np.array([gv.min(), gv.max()])
                ax.plot(xs, slope * xs + intercept, color=color, linewidth=2.4, zorder=4)
        ax.set_xlim(lo, hi)
        ax.set_ylim(lo, hi)
        ax.set_aspect("equal")
        _relabel_floor_ticks(ax)
        ax.set_title(antigen, fontsize=10.5, fontweight="bold")
        if row == len(antigens) - 1:
            ax.set_xlabel(f"-{focal_mut} affinity", fontsize=9.5)
        ax.set_ylabel(f"+{focal_mut} affinity", fontsize=9.5)
        for sp in ["top", "right"]:
            ax.spines[sp].set_visible(False)
        if row == 0:
            ax.legend(fontsize=7.5, loc="upper left", frameon=False)

    fig.suptitle(suptitle, fontsize=12.5, fontweight="bold", y=1.02)
    plt.tight_layout()

    save_figure(fig, Path(out_name).stem)
    plt.close(fig)


def _order_combo_palette(combo_muts):
    """Germline/single/double/triple palette for up to 3 combo_muts, reusing
    this project's ESTABLISHED ORDER0_COLOR/ORDER1_COLORS/ORDER2_COLORS/
    ORDER3_COLOR family (plot_three_way_order_violins.py) rather than a new
    ad hoc scheme -- germline=ORDER0_COLOR, each single=ORDER1_COLORS[i] by
    combo_muts index, each pair=ORDER2_COLORS[j] in
    itertools.combinations(range(k),2) order, the full triple=ORDER3_COLOR.
    Combo tuples preserve combo_muts' own order (matching how callers build
    them from `tuple(m for m in combo_muts if ...)`), so pair keys line up
    with the itertools.combinations index order used here."""
    k = len(combo_muts)
    assert k <= 3, "_order_combo_palette only covers up to 3 combo_muts (ORDER1/2_COLORS are length-3)"
    colors = {(): ORDER0_COLOR}
    labels = {(): "none"}
    for i, m in enumerate(combo_muts):
        colors[(m,)] = ORDER1_COLORS[i]
        labels[(m,)] = m
    if k >= 2:
        for j, (a, b) in enumerate(itertools.combinations(range(k), 2)):
            combo = (combo_muts[a], combo_muts[b])
            colors[combo] = ORDER2_COLORS[j]
            labels[combo] = "+".join(combo)
    if k == 3:
        combo = tuple(combo_muts)
        colors[combo] = ORDER3_COLOR
        labels[combo] = "+".join(combo)
    return colors, labels


def matched_pair_full_combo_scatter(df, antibody, focal_mut, combo_muts, out_name, suptitle, antigens=None,
                                     restrict_germline_at=None, restrict_mutant_at=None, axis_cond_label=None,
                                     panel_note=None, show_title=True):
    """Simple raw matched-pair genotype scatter (x=-focal_mut, y=+focal_mut,
    no panel split) but colored by the FULL combo of combo_muts (up to 3,
    2^len(combo_muts) states) using the germline/single/double/triple
    ORDER-family palette -- e.g. folding a conditioning mutation like N52K
    IN as part of the color scheme instead of splitting it into its own
    panel, so one glance at color position relative to the diagonal shows
    the whole story (a 3-mutation interaction/synergy pattern) without
    needing a separate delta-vs-delta plot. axis_cond_label: if a site is
    held fixed via restrict_mutant_at/restrict_germline_at (e.g. N52K held
    mutant throughout), pass its sign-notation label (e.g. "+N52K") to make
    that explicit on the axis text itself ("-L45P +N52K affinity") instead
    of leaving it implicit in the restriction alone. panel_note: alternative
    to axis_cond_label -- keeps the axis text plain and instead adds a
    small second line under each panel's antigen title (e.g. "+H35N"),
    when the restriction should be visible without altering the axis
    labels themselves."""
    k2i = kabat_to_site_index(antibody)
    antigens = antigens or list(ANTIGENS)
    combo_colors, combo_labels = _order_combo_palette(combo_muts)
    idx_by_name = {m: k2i[m] for m in combo_muts}
    cond_name, rest_muts = combo_muts[0], combo_muts[1:]
    c_idx = idx_by_name[cond_name]

    def combo_of(rem_tuple):
        return tuple(m for m in combo_muts if idx_by_name[m] in rem_tuple)

    panel_data = {}
    all_vals = []
    for row, antigen in enumerate(antigens):
        d = df[(df.antibody == antibody) & (df.antigen == antigen)].dropna(subset=[VALUE_COL_RAW]).copy()
        for m in (restrict_germline_at or []):
            d = d[d["site_seq"].str[k2i[m]] == GERMLINE]
        for m in (restrict_mutant_at or []):
            d = d[d["site_seq"].str[k2i[m]] != GERMLINE]
        merged, rem_idx = build_split_frame(d, antibody, antigen, focal_mut, cond_name, rest_muts, k2i)
        full_rem = merged.apply(lambda r: r["rem"] + ((c_idx,) if r["cont"] else ()), axis=1)
        combos = full_rem.apply(combo_of)
        panel_data[row] = (merged["germ_val"].values, merged["mat_val"].values, combos)
        all_vals.append(merged["germ_val"].values)
        all_vals.append(merged["mat_val"].values)
    lo, hi = _global_range(all_vals)
    x_label = f"-{focal_mut} {axis_cond_label} affinity" if axis_cond_label else f"-{focal_mut} affinity"
    y_label = f"+{focal_mut} {axis_cond_label} affinity" if axis_cond_label else f"+{focal_mut} affinity"

    fig, axes = plt.subplots(len(antigens), 1, figsize=(4.4, 3.8 * len(antigens)), dpi=170, squeeze=False)
    for row, antigen in enumerate(antigens):
        gv, mv, combos = panel_data[row]
        ax = axes[row, 0]
        point_colors = [combo_colors[c] for c in combos]
        _scatter_panel_multicolor(
            ax, gv, mv, point_colors, antigen, lo, hi,
            xlabel=x_label if row == len(antigens) - 1 else None,
            ylabel=y_label,
        )
        if panel_note:
            ax.set_title(f"{antigen} (n={len(gv)})\n{panel_note}", fontsize=10, fontweight="normal")

    plt.tight_layout()
    combo_counts = pd.concat([panel_data[row][2] for row in panel_data]).value_counts()
    _draw_combo_key(fig, axes, combo_muts, combo_colors, combo_counts)
    if show_title:
        fig.suptitle(suptitle, fontsize=12.5, fontweight="bold", y=1.03)

    save_figure(fig, Path(out_name).stem)
    plt.close(fig)
    print("Saved:", out)

## 6. Make the figures

In [ ]:
df = load_raw_binding_data()
k2i = kabat_to_site_index("NIH45-46")
s54g_idx, i51l_idx = k2i["S54G"], k2i["I51L"]

PERM_PANELS = [
    ("-S54G -I51L", lambda r, ri: s54g_idx not in r and i51l_idx not in r),
    ("+S54G -I51L", lambda r, ri: s54g_idx in r and i51l_idx not in r),
    ("-S54G +I51L", lambda r, ri: s54g_idx not in r and i51l_idx in r),
    ("+S54G +I51L", lambda r, ri: s54g_idx in r and i51l_idx in r),
]
PERM_COLORS = [ORDER0_COLOR, ORDER1_COLORS[0], ORDER1_COLORS[2], ORDER3_COLOR]

### Figure 1 - N52K germline (selectivity control)

In [ ]:
matched_pair_scatter_grid(
    df, "NIH45-46", "L45P", "N52K", PERM_PANELS,
    out_name="NIH4546_matched_pair_scatter_L45P_by_S54G_I51L_perm_N52Kgerm.png",
    suptitle="NIH45-46: L45P matched-pair genotype scatter (germline vs mature), "
             "N52K-GERMLINE only, S54G x I51L",
    remaining_muts=["S54G", "I51L"],
    colors=PERM_COLORS,
    cond_state=False,      # <- N52K held GERMLINE
)

### Figure 2 - N52K mutant

In [ ]:
matched_pair_scatter_grid(
    df, "NIH45-46", "L45P", "N52K", PERM_PANELS,
    out_name="NIH4546_matched_pair_scatter_L45P_by_S54G_I51L_perm_N52Kmut.png",
    suptitle="NIH45-46: L45P matched-pair genotype scatter (germline vs mature), "
             "N52K-mutant only, S54G x I51L",
    remaining_muts=["S54G", "I51L"],
    colors=PERM_COLORS,    # cond_state defaults to True -> N52K mutant
)

### Figure 3 - by N52K state, coloured by S54G × I51L combo

In [ ]:
cond_selectivity_by_combo_scatter(
    df, "NIH45-46", "L45P", "N52K", ["S54G", "I51L"],
    out_name="NIH4546_matched_pair_scatter_L45P_by_N52K_S54GI51Lcombo.png",
    suptitle="NIH45-46: L45P matched-pair scatter by N52K state, colored by S54G x I51L combo",
)